# AgentMorph Stage 1 - **Qwen2.5-7B**

Runs `Qwen2.5-7B` on Colab T4. Each model gets a clean Python process so VRAM fragmentation from prior loads doesn't carry over. All 5 per-model notebooks share one `--out-dir` on Drive; the manifest key includes the model id so there are no file collisions.

**Model size:** mid (~5.5 GB) - second smallest

**Default sweep:** `native + langgraph`. smolagents is intentionally excluded — its 30-tool system prompt blows the T4 context budget by step 3-4 (see §8 note).

**Order:** Llama-3.2-3B (canary) -> Qwen2.5-7B -> Llama-3.1-8B -> Gemma-2-9B -> Phi-4 (last; tightest on T4).

**Before you run:** Runtime -> Change runtime type -> **T4 GPU**. Have your HF token ready for §3.

## 1. GPU sanity check

In [1]:
import torch
assert torch.cuda.is_available(), "No CUDA visible. Runtime > Change runtime type > T4 GPU."
dev = torch.cuda.get_device_properties(0)
print(f"CUDA OK: {torch.cuda.get_device_name(0)} ({dev.total_memory / 1024**3:.1f} GB VRAM)")

CUDA OK: Tesla T4 (14.6 GB VRAM)


## 2. Clone / pull the repo

In [2]:
import os
REPO_URL = "https://github.com/Pranamya16/AgentMorph.git"
REPO_DIR = "/content/AgentMorph"
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch origin && git checkout main && git pull --ff-only
!cd {REPO_DIR} && git log -1 --oneline

Cloning into '/content/AgentMorph'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (93/93), done.
remote: Total 149 (delta 65), reused 122 (delta 45), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 427.63 KiB | 12.96 MiB/s, done.
Resolving deltas: 100% (65/65), done.
af42873 (HEAD -> main, origin/main, origin/HEAD) stage1: finish AgentDojo deliverable (hardened adapter + canary notebook)


## 3. HuggingFace auth

Paste your token, run the cell, then **clear the token from the cell before sharing this notebook.**

In [ ]:
from huggingface_hub import login
HF_TOKEN = "hf_REDACTED_ROTATED_2026_04_20"
login(token=HF_TOKEN)

## 4. Mount Drive + install extras

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
!pip install -q -e /content/AgentMorph[models,smolagents,langgraph,agentdojo]

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.4/192.4 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.7/155.7 kB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 85.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 635.9/635.9 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 350.5/350.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.3 MB/s 

## 5. Set the model for this notebook

Each notebook is named after the model it runs.

In [6]:
MODEL = "Qwen2.5-7B"
print("this notebook will run:", MODEL)

this notebook will run: Qwen2.5-7B


## 6. (Optional) wipe prior `Qwen2.5-7B` cells from the shared run

Prunes ONLY this model's rows from the shared `manifest.json` + deletes its JSONL files, leaving other models' progress untouched. Run after an adapter fix when you want to retry this model's trajectories from scratch.

In [7]:
import json, pathlib
RUN_DIR = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline")
manifest_path = RUN_DIR / "manifest.json"
traj_dir = RUN_DIR / "trajectories"

if manifest_path.exists():
    data = json.loads(manifest_path.read_text())
    before = len(data.get("completed", {}))
    data["completed"] = {k: v for k, v in data.get("completed", {}).items() if not k.startswith(MODEL + "|")}
    after = len(data["completed"])
    manifest_path.write_text(json.dumps(data, indent=2))
    print(f"manifest: {before} -> {after} entries (dropped {before - after} for {MODEL})")
else:
    print("no manifest - nothing to prune")

if traj_dir.exists():
    for p in traj_dir.glob(f"{MODEL}__*.jsonl"):
        p.unlink()
        print("deleted:", p.name)
else:
    print("no trajectory dir yet")

manifest: 120 -> 80 entries (dropped 40 for Qwen2.5-7B)
deleted: Qwen2.5-7B__smolagents__ecommerce.jsonl
deleted: Qwen2.5-7B__langgraph__ecommerce.jsonl


## 7. Smoke test - `Qwen2.5-7B` x native x 3 scenarios

Proves the model loads and tools + trajectory logger work end-to-end. Writes to `stage1_smoke_<MODEL>/` (separate from the baseline dir) so it doesn't pollute §8.

In [8]:
!python -m agentmorph.runner \
    --model {MODEL} --framework native --env ecommerce --n-scenarios 3 \
    --hf-cache-dir /content/drive/MyDrive/AgentMorph/hf_cache \
    --out-dir /content/drive/MyDrive/AgentMorph/runs/stage1_smoke_{MODEL}

2026-04-20 04:08:04.438326: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776658084.458490    6959 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776658084.465019    6959 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776658084.481306    6959 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776658084.481349    6959 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776658084.481354    6959 computation_placer.cc:177] computation placer alr

## 8. Framework sweep - `Qwen2.5-7B` x (native + langgraph) x ecommerce x 20 scenarios

Resumable. Writes into the shared `stage1_baseline/` directory.

**Why `native + langgraph` only (not smolagents):** smolagents injects all 30 tool descriptions into every turn's prompt, which on small open models snowballs input context to 20K+ tokens by step 3 and triggers CUDA OOM on a 14.6 GB T4. The smolagents adapter stays in the code — add `--framework smolagents` below to test it on larger / more tool-calling-capable models (Qwen2.5-7B, Gemma-2-9B, Phi-4).

In [9]:
import os
# Reduce CUDA allocator fragmentation on long multi-scenario runs.
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m agentmorph.runner \
    --model {MODEL} --framework native --framework langgraph --env ecommerce \
    --hf-cache-dir /content/drive/MyDrive/AgentMorph/hf_cache \
    --out-dir /content/drive/MyDrive/AgentMorph/runs/stage1_baseline

2026-04-20 04:14:14.903194: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1776658454.923415    8575 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1776658454.929624    8575 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1776658454.945646    8575 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776658454.945690    8575 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1776658454.945694    8575 computation_placer.cc:177] computation placer alr

## 9. Diagnostic - full error text for Qwen2.5-7B

Prints the complete (non-truncated) content of the first error step per framework. Paste back here if anything fails.

In [10]:
import json, pathlib
traj_dir = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline/trajectories")
for p in sorted(traj_dir.glob(f"{MODEL}__*.jsonl")):
    print(f"\n======= {p.name} =======")
    shown = False
    for line in p.open():
        t = json.loads(line)
        errs = [s for s in t["steps"] if s["kind"] == "error"]
        if errs and not shown:
            print(f"scenario: {t['scenario_id']}")
            print(f"steps: {len(t['steps'])}  final_answer: {t['final_answer']!r}")
            print("FULL ERROR TEXT:")
            print(errs[0]["content"])
            shown = True
            break
    if not shown:
        print("(no error steps in this file - all trajectories finished cleanly!)")


======= Qwen2.5-7B__langgraph__ecommerce.jsonl =======
scenario: eco_checkout_basic
steps: 1  final_answer: None
FULL ERROR TEXT:
langgraph adapter failed: KeyError: 'no product chef_knife_id'

======= Qwen2.5-7B__native__ecommerce.jsonl =======
scenario: eco_shop_kettle
steps: 10  final_answer: None
FULL ERROR TEXT:
max_steps exhausted without a FINAL answer


## 10. Finish-rate for `Qwen2.5-7B`

In [11]:
import json, collections, pathlib
traj_dir = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline/trajectories")
stats = collections.defaultdict(lambda: [0, 0])
for p in traj_dir.glob(f"{MODEL}__*.jsonl"):
    for line in p.open():
        t = json.loads(line)
        key = (t["model_id"], t["framework_id"])
        stats[key][0] += 1
        if t["final_answer"]:
            stats[key][1] += 1
print(f"=== {MODEL} finish rate ===")
for (m, f), (tot, fin) in sorted(stats.items()):
    pct = 100 * fin / tot if tot else 0
    print(f"  {f:10s}  finished {fin:2d}/{tot:2d}  ({pct:3.0f}%)")

=== Qwen2.5-7B finish rate ===
  langgraph   finished 12/20  ( 60%)
  native      finished  8/20  ( 40%)


## 11. Peek one trajectory from `Qwen2.5-7B`

In [12]:
import json, pathlib
traj_dir = pathlib.Path("/content/drive/MyDrive/AgentMorph/runs/stage1_baseline/trajectories")
for p in sorted(traj_dir.glob(f"{MODEL}__*.jsonl")):
    sample = p.open().readline()
    if not sample:
        continue
    t = json.loads(sample)
    print(f"\n======= {p.name} =======")
    print(f"scenario: {t['scenario_id']}")
    print(f"framework: {t['framework_id']}")
    print(f"steps: {len(t['steps'])}  wall_seconds: {t.get('wall_seconds') or 0:.1f}")
    print(f"final_answer: {t['final_answer']!r}\n")
    for s in t["steps"][:20]:
        tag = s["kind"]
        if tag == "tool_call":
            print(f"  [{tag}] {s['tool_name']}({s['tool_args']})")
        elif tag == "tool_result":
            print(f"  [{tag}] {s['tool_name']} -> {str(s['tool_output'])[:80]}")
        else:
            print(f"  [{tag}] {str(s.get('content', ''))[:200]}")


======= Qwen2.5-7B__langgraph__ecommerce.jsonl =======
scenario: eco_shop_kettle
framework: langgraph
steps: 6  wall_seconds: 12.6
final_answer: "I've added the Electric Kettle to your cart. The total in your cart is $39.00."

  [tool_call] search_products({'query': 'kettle', 'max_price': 50, 'limit': 1})
  [tool_result] search_products -> [{"id": "P020", "name": "Electric Kettle", "category": "kitchen", "price": 39.0,
  [tool_call] add_to_cart({'product_id': 'P020', 'quantity': 1})
  [tool_result] add_to_cart -> {"items": [{"product_id": "P020", "name": "Electric Kettle", "unit_price": 39.0,
  [thought] I've added the Electric Kettle to your cart. The total in your cart is $39.00.
  [final_answer] I've added the Electric Kettle to your cart. The total in your cart is $39.00.

======= Qwen2.5-7B__native__ecommerce.jsonl =======
scenario: eco_shop_kettle
framework: native
steps: 10  wall_seconds: 13.5
final_answer: None

  [thought] ```json
{"tool": "search_products", "arguments": {"qu